# TFTokenizer example

Fit TFTokenizer on chr19 (hg38, JASPAR 2026 motif hits)

In [ ]:
from pathlib import Path

TRAIN, CHROMS = ["chr19"], ["chr19", "chr22"]
DATA, OUT = Path("data/hits"), Path("tokens")
MIRROR = "https://hgdownload.soe.ucsc.edu"   # or the Europe mirror https://hgdownload-euro.soe.ucsc.edu

## 1. Download

Motif hits of the chosen chromosomes from the UCSC JASPAR track (hg38, JASPAR2026); tables already there are skipped.

In [ ]:
from tftokenizer import download

hits = dict(zip(CHROMS, download(CHROMS, DATA, mirror=MIRROR, log=print)))

## 2. Fit and tokenize


In [ ]:
import time
from tftokenizer import TFTokenizer

t = time.time()
tok = TFTokenizer().fit([(hits[c], c) for c in TRAIN])
OUT.mkdir(exist_ok=True)
tok.save(OUT / "model.npz")
print(f"fit on {TRAIN}: {time.time() - t:.0f} s, {len(tok.motifs)} motifs")
for chrom in CHROMS:
    t = time.time(); tok.tokenize(hits[chrom], chrom, OUT / chrom)
    print(f"tokenize {chrom}: {time.time() - t:.0f} s")

## 3. Output


In [ ]:
import numpy as np, zarr

for chrom in CHROMS:
    bins, tokens = np.load(OUT / chrom / "bins.npy"), zarr.open_array(str(OUT / chrom / "tokens.zarr"), mode="r")
    print(f"{chrom}: {tokens.shape[0]:,} tokens × {tokens.shape[1]} motifs, median length {np.median(np.diff(bins)):.0f} bp, "
          f"{1000 * tokens.shape[0] / bins[-1]:.1f} tokens / kb")